# EO Cavity temperature fit
Cavity temperature measured with spectrum analyzer

## Libraries and functions

In [13]:
from scipy.stats import norm
#from configuration import *
#from qm import SimulationConfig
#from tools import *
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal
from scipy.stats import norm
from scipy import interpolate
from scipy.optimize import minimize
from time import time
from scipy.optimize import curve_fit
import scipy.constants as cs
from datetime import datetime
from scipy.io import savemat, loadmat
#import pyvisa as visa
import os
import shutil
# import Lorentz_fit as Lorentz
import json
import glob
from tabulate import tabulate
from scipy.optimize import fsolve
import uncertainties.unumpy as unp
from uncertainties import ufloat
%matplotlib notebook

In [14]:
def fitRabi(t, A0, tau2, omega, phi, B0):
    return A0*np.exp(-t/tau2)*np.sin(omega*t + phi) + B0
def GammaThermalShotnoise(ktot, chi, nth): #From Rigetti paper, kappa with Amplitude decay
    #Superconducting qubit in a waveguide cavity with a coherence time approaching 0.1 ms
    #Using a qubit to measure photon-number statistics of a driven thermal oscillator
    return ktot/2 * np.real((np.sqrt((1+2*1j*chi/ktot)**2 + (8j * chi * nth / ktot))-1))
    #return ktot/2 * np.real((np.sqrt((1+2*1j*chi/ktot)**2 + (8j * chi * nth / ktot))-1))
    #return (1+2*1j*chi/ktot)**2 + (8j * chi * nth / ktot)
def GammaThermalShotnoiseSimple(ktot, chi, nth): #From Rigetti paper, kappa with Amplitude decay
    #Using a qubit to measure photon-number statistics of a driven thermal oscillator
    return ktot * chi**2/(ktot**2/4+chi**2)*nth
def GammaThermalShotnoise2(ktot, nth, N): #From Schoelkopf paper
    #N is the photon peak at frequency wq - N*chi
    #Photon shot noise dephasing in the strong-dispersive limit of circuit QED
    return ktot * ((nth + 1)*N + nth*(N + 1))
#Assume a (also decaying possible) cavity population and compare the increased decay to the T2 Echo
def T2Shotnoise(t, delta, ktot, chi, n, T2Echo, phi): #From Gambetta paper
    #Rapid Driven Reset of a Qubit Readout Resonator
    tau = (1-np.exp(-(ktot + 2j*chi)*t))/(ktot + 2j*chi)
    return 1/2*(1-np.imag(np.exp(-(1/T2Echo + 1j*delta)*t + (phi-2*n*chi*tau)*1j)))
#Cavity induced Ramsey oscillations
def T2Coh(t,chi,T2, n): # from Ammpendix E3 of Number-Resolved Photocounter for Propagating Microwave Mode R.
    return np.cos(n * np.sin(chi*t)) * np.exp( n * (np.cos(chi*t)-1)-t/T2)
def T2Thermal(t,chi,T2, nth): # from Ammpendix B1 of Dissipative Stabilization of Squeezing beyond 3 dB in a Microwave Mode
    #nth of the cavity
    return (nth*(1-np.cos(chi * t)) + 1) / (2*(1-np.cos(chi*t))*(nth+1)+1)*np.exp(-t/T2)
def BoseEinstein(f,T):
    return 1/(np.exp(cs.h*f/(cs.k*T))-1)
def Tphi(T1,T2):#this is valid if T1 and T2 have an exponential decay (T2 limited by 2T1 and e.g. white photon shot noise)
    #(1/f noise and several noise sources in general make a Gaussian noise statistics, so you have
    #T2 ~ exp(-t/2T1)*exp(-t**2/Tphi**2))
    return 1/(1/T2-1/(2*T1))
def TphiTrab(T1,Trab):
    return 1/(2/Trab-3/(2*T1))
def BoseEinsteinInverse(f,n):
    return cs.h*f/(cs.k*np.log(1/n+1))
def BoseEinsteinInverse_unp(f,n):
    return cs.h*f/(cs.k*unp.log(1/n+1))
def thermaldist(nmean, N):
    return nmean**N / (nmean+1)**(N+1)
def Poissondist(nmean,N):
    return np.exp(-nmean)*nmean**N / np.fact(N)
def gauss(x, mu, sigma, A):
    return A*np.exp(-(x-mu)**2/2/sigma**2)

def bimodal(x, mu1, sigma1, A1, mu2, sigma2, A2):
    return gauss(x,mu1,sigma1,A1)+gauss(x,mu2,sigma2,A2)

In [15]:
def T2CohwithScaling(t,t0,chi,T2, n, A):
    return T2Coh(t-t0, chi,T2,n)*A

def fitSdiffCoh(xdata, ydata, t0, chi, T2, n, A, **bounds):
    guess = [t0, chi, T2, n, A]
    bounds['boundslow'] = boundslow
    bounds['boundshigh'] = boundshigh
    full_output = curve_fit(T2CohwithScaling, xdata, ydata, guess, bounds = (boundslow,boundshigh), full_output=True)
    return full_output
    
def fit_expdecay(xdata, ydata, T2):
    guess = [T2, np.mean(ydata), np.abs(np.max(ydata)-np.min(ydata))]
    popt, pcov = curve_fit(exp, xdata, ydata, guess)
    return popt

## Main script

In [16]:
home_dir = r'.\Fig_4c\EOCav'

In [17]:
if os.path.isdir(home_dir):
    print('Folder found!')

SAoff20kHz = loadmat(home_dir + r'\Noise_JPAoff_20kHzBW_avg100000_7p5mK_55_data_230704_11h01m24s.mat')
SA10Hz20kHz = loadmat(home_dir + r'\Noise_10Hz_JPAOff_50kAvg_39_data_230705_19h24m25s.mat')
SA50Hz20kHz = loadmat(home_dir + r'\Noise_50Hz_JPAOff_50kAvg_44_data_230706_03h30m28s.mat')
SA250Hz20kHz = loadmat(home_dir + r'\Noise_250Hz_JPAOff_50kAvg_49_data_230706_14h49m57s.mat')
SA500Hz20kHz = loadmat(home_dir + r'\Noise_500Hz_JPAOff_50kAvg_50_data_230706_17h07m34s.mat')
SA1000Hz20kHz = loadmat(home_dir + r'\Noise_1kHz_JPAOff_50kAvg_52_data_230706_20h48m33s.mat')

# plt.figure(figsize = (9,4))
# plt.subplot(121)
# plt.plot(SAoff20kHz['Freq_trace_GHz_2'][0],SAoff20kHz['I_4'][0])
# plt.plot(SA10Hz20kHz['Freq_trace_GHz_2'][0],SA10Hz20kHz['I_4'][0])
# plt.plot(SA50Hz20kHz['Freq_trace_GHz_2'][0],SA50Hz20kHz['I_4'][0])
# plt.plot(SA250Hz20kHz['Freq_trace_GHz_2'][0],SA250Hz20kHz['I_4'][0])
# plt.plot(SA500Hz20kHz['Freq_trace_GHz_2'][0],SA500Hz20kHz['I_4'][0])
# plt.plot(SA1000Hz20kHz['Freq_trace_GHz_2'][0],SA1000Hz20kHz['I_4'][0])
# plt.title('PSD RBW = 20 kHz')
# plt.xlabel('f (GHz)')
# plt.ylabel('P (dBm/RBW)')
# axlims = plt.axis()
# plt.vlines([8.8065,8.8323],axlims[2],axlims[3], color = 'gray', linestyle = 'dashed')


Folder found!


In [18]:
# plt.figure(figsize = (9,4))
# norm20kHz = SAoff20kHz['I_4'][0]
# plt.subplot(121)
# plt.plot(SAoff20kHz['Freq_trace_GHz_2'][0],SAoff20kHz['I_4'][0]-norm20kHz,label='Baseline')
# plt.plot(SA10Hz20kHz['Freq_trace_GHz_2'][0],SA10Hz20kHz['I_4'][0]-norm20kHz,label='1st meas')
# plt.plot(SA50Hz20kHz['Freq_trace_GHz_2'][0],SA50Hz20kHz['I_4'][0]-norm20kHz,label='2nd meas')
# plt.xlabel('freq (GHz)')
# plt.ylabel('P (dB)')
# plt.title('Compare baseline to first two measurements; 20kHz RBW')
# plt.grid()
# plt.show()

In [19]:
def PHz(data,RBW):
    return 1/RBW*0.001*10**(data/10)
def PHz_unp(data,RBW):
    return 1/RBW*0.001*10**(data/10)

def Ndet(w,w0,eta,kappa,nb,nwg,nsys):
    return 4*kappa*eta*kappa*(1-eta)/(kappa**2+4*(w-w0)**2)*(nb-nwg) + nwg + nsys + 0.5
def Ndet_unp(w,w0,eta,kappa,nb,nwg,nsys):
    return 4*kappa*eta*kappa*(1-eta)/(kappa**2+4*(w-w0)**2)*(nb-nwg) + nwg + nsys + 0.5

def Nmode(eta,nb,nwg):
    return eta*nwg + (1-eta)*nb

In [20]:
fEO = 8.8071e9
kexEO = 3.26e6
kintEO = 6.23e6
kEO = kexEO+kintEO
etaEO = kexEO/kEO

fqu = 8.8323e9
kexqu = 1.02e6
kintqu = 0.38e6
kqu = kexqu + kintqu
etaqu = kexqu/kqu

In [21]:
Nmode(etaEO,0.001,0.001)
BoseEinstein(fEO,0.4)

0.5328162899187514

In [22]:
from scipy.optimize import curve_fit
import autograd

In [23]:
uncertainty_in_level_dB = 0.07 #dB, from Spectrum analyzer
uncertainty_in_level = 10**(uncertainty_in_level_dB/10)
print('uncertainty_in_level: ' + str(uncertainty_in_level))

nadd = 16.6*10**(1.5/10)
fitEOnoise = lambda w, nb, nwg: Ndet(w, fEO/1e9, etaEO, kEO/1e9, nb, nwg, nadd-0.5)
fitEOnoise_unp = lambda w, nb, nwg: Ndet_unp(w, fEO/1e9, etaEO, kEO/1e9, nb, nwg, nadd-0.5)
fitqunoise = lambda w, nb, nwg: Ndet(w, fqu/1e9, etaqu, kqu/1e9, nb, nwg, nadd-0.5)
norm20kHz = PHz(SAoff20kHz['I_4'][0],20e3)/nadd
fitparams = np.zeros((4,3))
p0 = [1.0,0.1]
lower_end = 0
upper_end = -1
full_output10Hz20kHz = curve_fit(fitEOnoise, SA10Hz20kHz['Freq_trace_GHz_2'][0][lower_end:upper_end], PHz(SA10Hz20kHz['I_4'][0][lower_end:upper_end],20e3)/norm20kHz[lower_end:upper_end], p0=p0, sigma=(PHz(SA10Hz20kHz['I_4'][0][lower_end:upper_end],20e3)/norm20kHz[lower_end:upper_end])*(uncertainty_in_level-1), absolute_sigma=True, full_output=True)
fitparams10Hz20kHz = full_output10Hz20kHz[0]
cov10Hz20kHz = full_output10Hz20kHz[1]
infodict10Hz20kHz = full_output10Hz20kHz[2:]

full_output50Hz20kHz = curve_fit(fitEOnoise, SA50Hz20kHz['Freq_trace_GHz_2'][0], PHz(SA50Hz20kHz['I_4'][0],20e3)/norm20kHz, p0=p0, sigma=(PHz(SA50Hz20kHz['I_4'][0],20e3)/norm20kHz)*(uncertainty_in_level-1), absolute_sigma=True, maxfev=10000, bounds = ([0,0],[nadd,nadd]), full_output=True)
full_output50Hz20kHz = curve_fit(fitEOnoise, SA50Hz20kHz['Freq_trace_GHz_2'][0], PHz(SA50Hz20kHz['I_4'][0],20e3)/norm20kHz, p0=p0, sigma=(PHz(SA50Hz20kHz['I_4'][0],20e3)/norm20kHz)*(uncertainty_in_level-1), absolute_sigma=True, full_output=True)
fitparams50Hz20kHz = full_output50Hz20kHz[0]
cov50Hz20kHz = full_output50Hz20kHz[1]
infodict50Hz20kHz = full_output50Hz20kHz[2:]

full_output250Hz20kHz = curve_fit(fitEOnoise, SA250Hz20kHz['Freq_trace_GHz_2'][0], PHz(SA250Hz20kHz['I_4'][0],20e3)/norm20kHz, p0=p0, sigma=(PHz(SA250Hz20kHz['I_4'][0],20e3)/norm20kHz)*(uncertainty_in_level-1), absolute_sigma=True, maxfev=10000, bounds = ([0,0],[nadd,nadd]), full_output=True)
full_output250Hz20kHz = curve_fit(fitEOnoise, SA250Hz20kHz['Freq_trace_GHz_2'][0], PHz(SA250Hz20kHz['I_4'][0],20e3)/norm20kHz, p0=p0, sigma=(PHz(SA250Hz20kHz['I_4'][0],20e3)/norm20kHz)*(uncertainty_in_level-1), absolute_sigma=True, full_output=True)
fitparams250Hz20kHz = full_output250Hz20kHz[0]
cov250Hz20kHz = full_output250Hz20kHz[1]
infodict250Hz20kHz = full_output250Hz20kHz[2:]

full_output500Hz20kHz = curve_fit(fitEOnoise, SA500Hz20kHz['Freq_trace_GHz_2'][0], PHz(SA500Hz20kHz['I_4'][0],20e3)/norm20kHz, p0=p0, sigma=(PHz(SA500Hz20kHz['I_4'][0],20e3)/norm20kHz)*(uncertainty_in_level-1), absolute_sigma=True, maxfev=10000, bounds = ([0,0],[nadd,nadd]), full_output=True)
full_output500Hz20kHz = curve_fit(fitEOnoise, SA500Hz20kHz['Freq_trace_GHz_2'][0], PHz(SA500Hz20kHz['I_4'][0],20e3)/norm20kHz, p0=p0, sigma=(PHz(SA500Hz20kHz['I_4'][0],20e3)/norm20kHz)*(uncertainty_in_level-1), absolute_sigma=True, full_output=True)
fitparams500Hz20kHz = full_output500Hz20kHz[0]
cov500Hz20kHz = full_output500Hz20kHz[1]
infodict500Hz20kHz = full_output500Hz20kHz[2:]

full_output1000Hz20kHz = curve_fit(fitEOnoise, SA1000Hz20kHz['Freq_trace_GHz_2'][0], PHz(SA1000Hz20kHz['I_4'][0],20e3)/norm20kHz, p0=p0, sigma=(PHz(SA1000Hz20kHz['I_4'][0],20e3)/norm20kHz)*(uncertainty_in_level-1), absolute_sigma=True, maxfev=10000, bounds = ([0,0],[nadd,nadd]), full_output=True)
full_output1000Hz20kHz = curve_fit(fitEOnoise, SA1000Hz20kHz['Freq_trace_GHz_2'][0], PHz(SA1000Hz20kHz['I_4'][0],20e3)/norm20kHz, p0=p0, sigma=(PHz(SA1000Hz20kHz['I_4'][0],20e3)/norm20kHz)*(uncertainty_in_level-1), absolute_sigma=True, maxfev=10000, full_output=True)
fitparams1000Hz20kHz = full_output1000Hz20kHz[0]
cov1000Hz20kHz = full_output1000Hz20kHz[1]
infodict1000Hz20kHz = full_output1000Hz20kHz[2:]

######################################## Trying by adding uncertainties to the spectrum measurement
SA10Hz20kHz_unp = unp.uarray(SA10Hz20kHz['I_4'][0],0.01)
PHz10Hz20kHz_unp = PHz_unp(SA10Hz20kHz_unp,20e3)/norm20kHz
full_output10Hz20kHz_unp = curve_fit(fitEOnoise_unp, SA10Hz20kHz['Freq_trace_GHz_2'][0], unp.nominal_values(PHz10Hz20kHz_unp), p0=p0, full_output=True,sigma=unp.std_devs(PHz10Hz20kHz_unp))
fitparams10Hz20kHz_unp = full_output10Hz20kHz_unp[0]
cov10Hz20kHz_unp = full_output10Hz20kHz_unp[1]
infodict10Hz20kHz_unp = full_output10Hz20kHz_unp[2:]
############################################

# fig, ax  = plt.subplots(1,1, figsize = (9,4))
# ax.plot(SA10Hz20kHz['Freq_trace_GHz_2'][0],PHz(SA10Hz20kHz['I_4'][0],20e3)/norm20kHz, color = 'C0')
# ax.plot(SA10Hz20kHz['Freq_trace_GHz_2'][0],fitEOnoise(SA10Hz20kHz['Freq_trace_GHz_2'][0], *fitparams10Hz20kHz), linestyle = '-', color = 'black',label  = np.round(BoseEinsteinInverse(fEO,Nmode(etaEO,*fitparams10Hz20kHz)),3))
# ax.plot(SA50Hz20kHz['Freq_trace_GHz_2'][0],PHz(SA50Hz20kHz['I_4'][0],20e3)/norm20kHz, color = 'C1', label  = np.round(BoseEinsteinInverse(fEO,Nmode(etaEO,*fitparams50Hz20kHz)),3))
# ax.plot(SA50Hz20kHz['Freq_trace_GHz_2'][0],fitEOnoise(SA50Hz20kHz['Freq_trace_GHz_2'][0], *fitparams50Hz20kHz), color = 'black', label = np.round(fitparams50Hz20kHz,2))
# ax.plot(SA250Hz20kHz['Freq_trace_GHz_2'][0],PHz(SA250Hz20kHz['I_4'][0],20e3)/norm20kHz, color = 'C2', label  = np.round(BoseEinsteinInverse(fEO,Nmode(etaEO,*fitparams250Hz20kHz)),3))
# ax.plot(SA250Hz20kHz['Freq_trace_GHz_2'][0],fitEOnoise(SA250Hz20kHz['Freq_trace_GHz_2'][0], *fitparams250Hz20kHz), color = 'black', label = np.round(fitparams250Hz20kHz,2))
# ax.plot(SA500Hz20kHz['Freq_trace_GHz_2'][0],PHz(SA500Hz20kHz['I_4'][0],20e3)/norm20kHz, color = 'C3', label  = np.round(BoseEinsteinInverse(fEO,Nmode(etaEO,*fitparams500Hz20kHz)),3))
# ax.plot(SA500Hz20kHz['Freq_trace_GHz_2'][0],fitEOnoise(SA500Hz20kHz['Freq_trace_GHz_2'][0], *fitparams500Hz20kHz), color = 'black', label  = np.round(fitparams500Hz20kHz,2))
# ax.plot(SA1000Hz20kHz['Freq_trace_GHz_2'][0],PHz(SA1000Hz20kHz['I_4'][0],20e3)/norm20kHz, color = 'C4', label  = np.round(BoseEinsteinInverse(fEO,Nmode(etaEO,*fitparams1000Hz20kHz)),3))
# ax.plot(SA1000Hz20kHz['Freq_trace_GHz_2'][0],fitEOnoise(SA1000Hz20kHz['Freq_trace_GHz_2'][0], *fitparams1000Hz20kHz), color = 'black', label  = np.round(fitparams1000Hz20kHz,2))
# ax.set_title('PSD RBW = 20 kHz')
# ax.set_xlabel('f (GHz)')
# ax.set_ylabel('added noise')
# axlims = ax.get_ylim()
# ax.vlines([8.8065,8.8323],axlims[0],axlims[1], color = 'gray', linestyle = 'dashed')
# ax.legend()



nb10Hz_unp = ufloat(fitparams10Hz20kHz_unp[0],np.sqrt(np.diag(cov10Hz20kHz_unp))[0]) 
nwg10Hz_unp = ufloat(fitparams10Hz20kHz_unp[1],np.sqrt(np.diag(cov10Hz20kHz_unp))[1])
nb10Hz = ufloat(fitparams10Hz20kHz[0],np.sqrt(np.diag(cov10Hz20kHz))[0]) 
nwg10Hz = ufloat(fitparams10Hz20kHz[1],np.sqrt(np.diag(cov10Hz20kHz))[1])
nb50Hz = ufloat(fitparams50Hz20kHz[0],np.sqrt(np.diag(cov50Hz20kHz))[0]) 
nwg50Hz = ufloat(fitparams50Hz20kHz[1],np.sqrt(np.diag(cov50Hz20kHz))[1])
nb250Hz = ufloat(fitparams250Hz20kHz[0],np.sqrt(np.diag(cov250Hz20kHz))[0]) 
nwg250Hz = ufloat(fitparams250Hz20kHz[1],np.sqrt(np.diag(cov250Hz20kHz))[1])
nb500Hz = ufloat(fitparams500Hz20kHz[0],np.sqrt(np.diag(cov500Hz20kHz))[0]) 
nwg500Hz = ufloat(fitparams500Hz20kHz[1],np.sqrt(np.diag(cov500Hz20kHz))[1])
nb1000Hz = ufloat(fitparams1000Hz20kHz[0],np.sqrt(np.diag(cov1000Hz20kHz))[0]) 
nwg1000Hz = ufloat(fitparams1000Hz20kHz[1],np.sqrt(np.diag(cov1000Hz20kHz))[1])


TcavEOoff = np.nan
Tcav10Hz_unp = BoseEinsteinInverse_unp(fEO,Nmode(etaEO,nb10Hz_unp, nwg10Hz_unp))
Tcav10Hz = BoseEinsteinInverse_unp(fEO,Nmode(etaEO,nb10Hz, nwg10Hz))
Tcav50Hz = BoseEinsteinInverse_unp(fEO,Nmode(etaEO,nb50Hz, nwg50Hz))
Tcav250Hz = BoseEinsteinInverse_unp(fEO,Nmode(etaEO,nb250Hz, nwg250Hz))
Tcav500Hz = BoseEinsteinInverse_unp(fEO,Nmode(etaEO,nb500Hz, nwg500Hz))
Tcav1000Hz = BoseEinsteinInverse_unp(fEO,Nmode(etaEO,nb1000Hz, nwg1000Hz))

print('Tcav10Hz: ' + str(Tcav10Hz))
print('Tcav50Hz: ' + str(Tcav50Hz))
print('Tcav250Hz: ' + str(Tcav250Hz))
print('Tcav500Hz: ' + str(Tcav500Hz))
print('Tcav1000Hz: '+ str(Tcav1000Hz))

# plt.tight_layout()

uncertainty_in_level: 1.0162486928706955
Tcav10Hz: 0.139+/-0.022
Tcav50Hz: 0.445+/-0.012
Tcav250Hz: 0.947+/-0.012
Tcav500Hz: 1.190+/-0.012
Tcav1000Hz: 1.626+/-0.012
